#### 패키지 호출

In [ ]:
from com_fun import *
import geopandas as gpd


#### 상품권 구매량 + 구매자 분석 데이터셋

In [ ]:
# 분석 데이터 불러오기
merge_gift_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_gift_user.pkl')
merge_gift_user02 = merge_gift_user.copy()

# 회원정보 탈퇴 삭제
merge_gift_user02 = merge_gift_user02[merge_gift_user02['회원상태'] != '탈퇴']

# 상품권 항목만 선택
# merge_gift_user02 = merge_gift_user02[merge_gift_user02['수단구분'] == '상품권']

# 가입연도 추가
merge_gift_user02['가입일자_연도'] = pd.to_datetime(merge_gift_user02['가입일자']).dt.year

# 구매금액 분포 추가
merge_gift_user02['만원단위'] = (merge_gift_user02['구매금액'] // 10000) * 10000
merge_gift_user02['구매금액_구간'] = merge_gift_user02['만원단위'].apply(lambda x: '20만원이상' if x >= 200000 else '1만원이하' if x == 0 else x)

# 할인율 추가
merge_gift_user02['할인율'] = merge_gift_user02['할인금액'] / merge_gift_user02['총금액']

# 필요없는 컬럼 제외
merge_gift_user02.drop(['지자체명', '구매_지급일자', '구매_지급시간', '가입일자','생년월일', '만원단위'], axis = 1, inplace = True)

# 정렬
merge_gift_user02 = merge_gift_user02[['구매_지급번호', '회원번호', '시도명', '시군구명', '주거래지역', '수단구분', 
                                       '구매_지급일', '구매_연도', '구매_월', '구매_일', '구매금액', '할인금액', '총금액', '구매금액_구간', '할인율',
                                       '회원상태', '가입일자_연도', '성별', '연령', '연령대']]

# 구매금액 0원 제외
merge_gift_user02 = merge_gift_user02[merge_gift_user02['구매금액'] != 0]

In [ ]:
# 가입시기별 회원 가입수 확인
tmp_df = merge_gift_user02[['가입일자_연도', '구매_연도', '회원번호']].groupby(['가입일자_연도', '구매_연도']).count().reset_index()
tmp_df.pivot_table(columns = '구매_연도', index = '가입일자_연도', values = '회원번호')

In [ ]:
# 지역 및 인구 기준 시각화 분석 셋 생성
view_df_buy01 = merge_gift_user02[['가입일자_연도', '구매_연도', '구매_월', '시도명', '시군구명', '회원상태', '성별', '연령대' ,'구매금액_구간', '구매금액']].\
                groupby(['가입일자_연도', '구매_연도', '구매_월', '시도명', '시군구명', '회원상태', '성별', '연령대' , '구매금액_구간']).agg({'sum', 'count'}).reset_index()
view_df_buy01.columns = view_df_buy01.columns.droplevel(1)
view_df_buy01.columns = ['가입일자_연도', '구매_연도', '구매_월', '시도명', '시군구명', '회원상태', '성별', '연령대', '구매금액_구간', '구매금액_합계', '구매건수_합계']

# 성별 값 변환
view_df_buy01['성별'] = view_df_buy01['성별'].replace({1 : '남자', 2 : '여자'})

# 연령대 변환
view_df_buy01['연령대'] = view_df_buy01['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})
view_df_buy01.to_excel('../P1_Data/02.Pretreatment/excel/view_df_buy01.xlsx', index = False)

In [ ]:
# 지역기준 시각화 분석 셋 생성
view_df_buy02 = merge_gift_user02[['주거래지역', '연령대','구매금액_구간', '구매금액']].\
                groupby(['주거래지역', '연령대','구매금액_구간']).agg({'sum', 'count'}).reset_index()
view_df_buy02.columns = view_df_buy02.columns.droplevel(1)
view_df_buy02.columns = ['주거래지역', '연령대','구매금액_구간', '구매금액_합계', '구매건수_합계']

# 연령대 변환
view_df_buy02['연령대'] = view_df_buy02['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})
view_df_buy02.to_excel('../P1_Data/02.Pretreatment/excel/view_df_buy02.xlsx', index = False)

In [ ]:
# 상위 10% 구매자 분석을 위한 분석 셋 생성
view_df_buy03 = merge_gift_user02.copy()
view_df_buy03 = view_df_buy03[['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대', '구매금액']].\
                groupby(['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대']).agg({'sum', 'count'}).reset_index()
view_df_buy03.columns = view_df_buy03.columns.droplevel(1)
view_df_buy03.columns = ['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대', '구매건수_합계', '구매금액_합계']

# '구매금액_합계' 기준으로 상위 10%에 해당하는 값을 가진 행을 찾기
top_10_percent = view_df_buy03['구매금액_합계'].quantile(0.9)

# 새로운 컬럼 추가 및 조건에 따라 True 또는 False 값 할당
view_df_buy03['구매금액_상위_10%'] = view_df_buy03['구매금액_합계'] >= top_10_percent

# 성별 값 변환
view_df_buy03['성별'] = view_df_buy03['성별'].replace({1 : '남자', 2 : '여자'})

# 연령대 변환
view_df_buy03['연령대'] = view_df_buy03['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})
view_df_buy03.to_excel('../P1_Data/02.Pretreatment/excel/view_df_buy03.xlsx', index = False)

In [ ]:
# 상위 10% 구매자 분석을 위한 분석 셋 생성
view_df_buy04 = merge_gift_user02.copy()
view_df_buy04 = view_df_buy04[['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대', '구매금액']].\
                groupby(['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대']).agg({'sum', 'count'}).reset_index()
view_df_buy04.columns = view_df_buy04.columns.droplevel(1)
view_df_buy04.columns = ['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대', '구매건수_합계', '구매금액_합계']

# 등간격 5% 단위로 상위 그룹 분리 및 새로운 컬럼 추가
view_df_buy04['분위수'] = pd.qcut(view_df_buy04['구매금액_합계'], q=[i/10.0 for i in range(11)], labels=False, duplicates='drop')

# 구간의 경계값 확인
bin_edges = pd.qcut(view_df_buy04['구매금액_합계'], q=[i/10.0 for i in range(11)], duplicates='drop')#.categories
print("구간의 경계값:")
print(bin_edges.value_counts())

# 분위수 결과 저장
view_df_buy04.to_excel('../P1_Data/02.Pretreatment/excel/view_df_buy04.xlsx', index = False)

In [ ]:
# 상위 10% 구매자 분석을 위한 분석 셋 생성
view_df_buy04 = merge_gift_user02.copy()
view_df_buy04 = view_df_buy04[['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대', '구매금액']].\
                groupby(['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대']).agg({'sum', 'count'}).reset_index()
view_df_buy04.columns = view_df_buy04.columns.droplevel(1)
view_df_buy04.columns = ['회원번호', '가입일자_연도', '주거래지역', '성별', '연령', '연령대', '구매건수_합계', '구매금액_합계']
view_df_buy05 = view_df_buy04[view_df_buy04['주거래지역'] == '성남시']

#### 상품권 이용량 + 가맹점 정보 + 사용자정보 분석 데이터셋

In [ ]:
merge_usage_store_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_usage_store_user.pkl')
# 가입연도 추가
merge_usage_store_user['결제일자_연도'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.year
merge_usage_store_user['결제일자_월'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.month

merge_usage_store_user = merge_usage_store_user[['가맹점ID', '가맹점명', '업종', '시도명', '시군구명', '회원번호', '성별', '연령대', '결제일자_연도', '결제일자_월', '결제금액']]
merge_usage_store_user.head()

In [ ]:
# 분석을 위한 분석 셋 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '회원번호']].\
                           groupby(['시도명', '시군구명']).agg({'count'}).reset_index()
merge_usage_store_user01.columns = merge_usage_store_user01.columns.droplevel(1)
merge_usage_store_user01.columns = ['시도명', '시군구명', '결제회원']
merge_usage_store_user01.head()

In [ ]:
# 분석을 위한 분석 셋 생성
merge_usage_store_user02 = merge_usage_store_user.copy()
merge_usage_store_user02 = merge_usage_store_user02[['시도명', '시군구명', '결제금액']].\
                           groupby(['시도명', '시군구명']).agg({'sum'}).reset_index()
merge_usage_store_user02.columns = merge_usage_store_user02.columns.droplevel(1)
merge_usage_store_user02.columns = ['시도명', '시군구명', '결제금액']
merge_usage_store_user02.head()

In [ ]:
merge_view_df_buy02 = merge_usage_store_user01.merge(merge_usage_store_user02, on = ['시도명', '시군구명'])
merge_view_df_buy02['1인당_결제금액'] = merge_view_df_buy02['결제금액'] / merge_view_df_buy02['결제회원']
merge_view_df_buy02 = merge_view_df_buy02.astype({'1인당_결제금액' : 'int64', '결제금액' : 'int64'})
merge_view_df_buy02

#### 결제내역 + 업종 + 사용자정보

In [ ]:
# 상품권 결제 정보 불러오기
merge_usage_store_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_usage_store_user.pkl')
# 가입연도 추가
merge_usage_store_user['결제일자_연도'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.year
merge_usage_store_user['결제일자_월'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.month

In [ ]:
# 분석을 위한 분석 셋 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['업종', '시도명', '시군구명', '성별', '연령대', '결제일자_연도', '결제일자_월', '결제금액']].\
                           groupby(['업종', '시도명', '시군구명', '성별', '연령대', '결제일자_연도', '결제일자_월']).agg({'sum', 'count'}).reset_index()
merge_usage_store_user01.columns = merge_usage_store_user01.columns.droplevel(1)
merge_usage_store_user01.columns = ['업종', '시도명', '시군구명', '성별', '연령대', '결제일자_연도', '결제일자_월', '소비건수_합계', '소비금액_합계']

# 성별 값 변환
merge_usage_store_user01['성별'] = merge_usage_store_user01['성별'].replace({1 : '남자', 2 : '여자'})

# 연령대 변환
merge_usage_store_user01['연령대'] = merge_usage_store_user01['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})
merge_usage_store_user01.to_excel('../P1_Data/02.Pretreatment/excel/view_df_usage01.xlsx', index = False)

In [ ]:
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['회원번호', '결제금액']].groupby(['회원번호']).agg({'sum'}).reset_index()
merge_usage_store_user01.columns = merge_usage_store_user01.columns.droplevel(1)
merge_usage_store_user01

In [ ]:
merge_usage_store_user01['분위수'] = pd.qcut(merge_usage_store_user01['결제금액'], q=[i/10.0 for i in range(11)], labels=False)#, duplicates='drop')
merge_usage_store_user01.to_excel('../P3_Result/01.Excel/결제금액_회원별_.xlsx', index = False)

In [ ]:
merge_usage_store_user01 = pd.read_excel('../P3_Result/01.Excel/결제금액_회원별_.xlsx')
merge_usage_store_user01['비율누적합'] = merge_usage_store_user01.비율.cumsum()
merge_usage_store_user01['결제금액_10분위수'] = pd.qcut(merge_usage_store_user01['결제금액'], q=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], labels=False)
merge_usage_store_user01.결제금액_10분위수.value_counts()

In [ ]:
# '결제금액'이라는 컬럼이 있다고 가정합니다.
# 실제 컬럼 이름에 맞게 수정해주세요.
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
quantile_values = merge_usage_store_user01['결제금액'].quantile(quantiles)

# 새로운 컬럼을 추가합니다.
merge_usage_store_user01['결제금액_10분위수'] = pd.cut(merge_usage_store_user01['결제금액'], bins=quantile_values, labels=False, include_lowest=True)
merge_usage_store_user01[['분위수', '결제금액']].groupby(['분위수']).agg({'mean'})

In [ ]:
# 지역별 성별 연령대별 이용고객 추출
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['결제일자_연도', '시도명', '시군구명', '성별', '연령대', '회원번호']].drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)
merge_usage_store_user01[['분위수', '회원번호']].groupby(['분위수']).agg({'mean'})

In [ ]:
# 지역별 성별 연령대별 이용고객 추출
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['결제일자_연도', '시도명', '시군구명', '성별', '연령대', '회원번호']].drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)
merge_usage_store_user01 = merge_usage_store_user01.value_counts(['결제일자_연도', '시도명', '시군구명', '성별', '연령대']).reset_index()

# 성별 값 변환
merge_usage_store_user01['성별'] = merge_usage_store_user01['성별'].replace({1 : '남자', 2 : '여자'})

# 연령대 변환
merge_usage_store_user01['연령대'] = merge_usage_store_user01['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})

# 60대 이상 중복 묶음
merge_usage_store_user01 = merge_usage_store_user01.groupby(['결제일자_연도', '시도명', '시군구명', '성별', '연령대']).agg({'sum'}).reset_index()
merge_usage_store_user01.columns = merge_usage_store_user01.columns.droplevel(1)
merge_usage_store_user01.columns = ['결제일자_연도', '시도명', '시군구명', '성별', '연령대', 'count']

# 특정지역의 성별 및 연령대별 소비인구 추출
merge_usage_store_user01[(merge_usage_store_user01['시군구명'] == '아산시') | (merge_usage_store_user01['시군구명'] == '군산시') | (merge_usage_store_user01['시군구명'] == '시흥시') | (merge_usage_store_user01['시군구명'] == '성남시')].to_excel('../P3_Result/01.Excel/특정지역_시점별_성별_연령대별_소비인구.xlsx', index = False)

In [ ]:
# 분석을 위한 분석 셋 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '성별', '연령대', '회원번호']].drop_duplicates(subset=None, keep='first', inplace=False, ignore_index=False)
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '성별', '연령대', '회원번호']].\
                           groupby(['시도명', '시군구명', '성별', '회원번호']).agg({'count'}).reset_index()

In [ ]:
# 분석을 위한 분석 셋 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['업종', '시도명', '시군구명', '성별', '연령대', '회원번호']].\
                           groupby(['업종', '시도명', '시군구명', '성별', '회원번호']).agg({'count'}).reset_index()
merge_usage_store_user01

#### 상품권 할인율별 판매차이 분석

In [ ]:
merge_gift_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_gift_user.pkl')
merge_gift_user

In [ ]:
merge_gift_user02 = merge_gift_user.copy()
# 회원정보 탈퇴 삭제
merge_gift_user02 = merge_gift_user02[merge_gift_user02['회원상태'] != '탈퇴']

# 상품권 항목만 선택
# merge_gift_user02 = merge_gift_user02[merge_gift_user02['수단구분'] == '상품권']

# 가입연도 추가
merge_gift_user02['가입일자_연도'] = pd.to_datetime(merge_gift_user02['가입일자']).dt.year

# 구매금액 분포 추가
merge_gift_user02['만원단위'] = (merge_gift_user02['구매금액'] // 10000) * 10000
merge_gift_user02['구매금액_구간'] = merge_gift_user02['만원단위'].apply(lambda x: '20만원이상' if x >= 200000 else '1만원이하' if x == 0 else x)

# 할인율 추가
merge_gift_user02['할인율'] = merge_gift_user02['할인금액'] / merge_gift_user02['총금액']

# 필요없는 컬럼 제외
merge_gift_user02.drop(['지자체명', '구매_지급일자', '구매_지급시간', '가입일자','생년월일', '만원단위'], axis = 1, inplace = True)

# 정렬
merge_gift_user02 = merge_gift_user02[['구매_지급번호', '회원번호', '시도명', '시군구명', '주거래지역', '수단구분', 
                                       '구매_지급일', '구매_연도', '구매_월', '구매_일', '구매금액', '할인금액', '총금액', '구매금액_구간', '할인율',
                                       '회원상태', '가입일자_연도', '성별', '연령', '연령대']]

# 구매금액 0원 제외
merge_gift_user02 = merge_gift_user02[merge_gift_user02['구매금액'] != 0]

In [ ]:
# 성별 값 변환
merge_gift_user02['성별'] = merge_gift_user02['성별'].replace({1 : '남자', 2 : '여자'})

# 연령대 변환
merge_gift_user02['연령대'] = merge_gift_user02['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})
merge_gift_user02.to_pickle('../P1_Data/02.Pretreatment/pkl/pre_merge_gift_user.pkl')

In [ ]:
# 지역기준 시각화 분석 셋 생성
view_df_buy02 = merge_gift_user02[['구매_연도', '구매_월', '시도명', '시군구명', '할인율', '성별', '연령대','구매_지급번호']].\
                groupby(['구매_연도', '구매_월', '시도명', '시군구명', '성별', '연령대', '할인율']).agg({'count'}).reset_index()
view_df_buy02.columns = view_df_buy02.columns.droplevel(1)
view_df_buy02.columns = ['구매_연도', '구매_월', '시도명', '시군구명', '성별', '연령대', '할인율', '구매건수_합계']
view_df_buy02.to_excel('../P3_Result/01.Excel/연도별_지역별_할인율_구매횟수.xlsx', index = False)
view_df_buy02

In [ ]:
# 지역기준 시각화 분석 셋 생성
view_df_buy02 = merge_gift_user02[['구매_연도', '구매_월', '시도명', '시군구명', '할인율','구매금액']].\
                groupby(['구매_연도', '구매_월', '시도명', '시군구명', '할인율']).agg({'sum'}).reset_index()
view_df_buy02.columns = view_df_buy02.columns.droplevel(1)
view_df_buy02.columns = ['구매_연도', '구매_월', '시도명', '시군구명', '할인율', '구매금액_합계']
view_df_buy02.to_excel('../P3_Result/01.Excel/연도별_지역별_할인율_구매금액.xlsx', index = False)
view_df_buy02

In [ ]:
view_df_buy03 = view_df_buy02[['시도명', '시군구명', '할인율', '구매건수_합계']].groupby(['시도명', '시군구명', '할인율']).agg({'sum'}).reset_index()
view_df_buy03.columns = view_df_buy03.columns.droplevel(1)
view_df_buy03.columns = ['시도명', '시군구명', '할인율', '구매횟수']
view_df_buy03

In [ ]:
view_df_buy03 = view_df_buy02[['할인율', '구매건수_합계']].groupby(['할인율']).agg({'sum'}).reset_index()
view_df_buy03.columns = view_df_buy03.columns.droplevel(1)
view_df_buy03.columns = ['할인율', '구매횟수']
view_df_buy03

#### 지역별 집계 GIS 시각화 데이터 생성

In [ ]:
# gis 지역 데이터 불러오기
area_df = gpd.read_file('../P1_Data/01.External/행정경계/Z_SOP_BND_SIGUNGU_PG/Z_SOP_BND_SIGUNGU_PG.shp', encoding = 'cp949')
area_df = area_df.drop(['BASE_DATE', 'SIGUNGU_CD', 'OBJECTID'], axis = 1)
area_df.rename(columns = {'SIGUNGU_NM' : '시군구명'}, inplace = True)

# 띄어쓰기를 기준으로 값을 수정하는 함수 정의
def remove_after_space(text):
    return text.split()[0]

# 데이터프레임의 해당 열에 함수를 적용하여 값 수정
area_df['시군구명'] = area_df['시군구명'].apply(remove_after_space)

from shapely.geometry import MultiPolygon

# GeoDataFrame을 iterate하면서 기하학적 오류를 수정합니다.
for index, row in area_df.iterrows():
    if isinstance(row.geometry, MultiPolygon):
        # MultiPolygon인 경우 buffer(0) 메서드를 적용합니다.
        new_geom = row.geometry.buffer(0)
        area_df.at[index, 'geometry'] = new_geom
    else:
        # 단일 Polygon인 경우 buffer(0) 메서드를 적용합니다.
        area_df.at[index, 'geometry'] = row.geometry.buffer(0)

# 같은 시군구명 합치기
area_df = area_df.dissolve(by='시군구명', aggfunc='sum')

In [ ]:
# 상품권 구매 데이터 불러오기
merge_gift_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_gift_user.pkl')

merge_gift_user02 = merge_gift_user.copy()
# 회원정보 탈퇴 삭제
merge_gift_user02 = merge_gift_user02[merge_gift_user02['회원상태'] != '탈퇴']

# 상품권 항목만 선택
# merge_gift_user02 = merge_gift_user02[merge_gift_user02['수단구분'] == '상품권']

# 가입연도 추가
merge_gift_user02['가입일자_연도'] = pd.to_datetime(merge_gift_user02['가입일자']).dt.year

# 구매금액 분포 추가
merge_gift_user02['만원단위'] = (merge_gift_user02['구매금액'] // 10000) * 10000
merge_gift_user02['구매금액_구간'] = merge_gift_user02['만원단위'].apply(lambda x: '20만원이상' if x >= 200000 else '1만원이하' if x == 0 else x)

# 할인율 추가
merge_gift_user02['할인율'] = merge_gift_user02['할인금액'] / merge_gift_user02['총금액']

# 필요없는 컬럼 제외
merge_gift_user02.drop(['지자체명', '구매_지급일자', '구매_지급시간', '가입일자','생년월일', '만원단위'], axis = 1, inplace = True)

# 정렬
merge_gift_user02 = merge_gift_user02[['구매_지급번호', '회원번호', '시도명', '시군구명', '주거래지역', '수단구분', 
                                       '구매_지급일', '구매_연도', '구매_월', '구매_일', '구매금액', '할인금액', '총금액', '구매금액_구간', '할인율',
                                       '회원상태', '가입일자_연도', '성별', '연령', '연령대']]

# 구매금액 0원 제외
merge_gift_user02 = merge_gift_user02[merge_gift_user02['구매금액'] != 0]

In [ ]:
# 지역별 구매회원 데이터 생성
view_df_buy05 = merge_gift_user02[['시도명', '시군구명', '회원번호']].drop_duplicates('회원번호')
view_df_buy05 = view_df_buy05.value_counts(['시도명', '시군구명']).reset_index()
view_df_buy05.rename(columns = {'count' : '구매회원'}, inplace = True)

# 지역별 구매금액 데이터 생성
view_df_buy06 = merge_gift_user02[['시도명', '시군구명', '구매금액']].\
                groupby(['시도명', '시군구명']).agg({'sum'}).reset_index()
view_df_buy06.columns = view_df_buy06.columns.droplevel(1)
view_df_buy06.columns = ['시도명', '시군구명', '구매금액']

# 구매회원 구매금액 결합
merge_view_df_buy01 = view_df_buy05.merge(view_df_buy06, on = ['시도명', '시군구명'])
merge_view_df_buy01['1인당_구매금액'] = merge_view_df_buy01['구매금액'] / merge_view_df_buy01['구매회원']
merge_view_df_buy01 = merge_view_df_buy01.astype({'1인당_구매금액' : 'int64'})

In [ ]:
# 상품권 결제 정보 불러오기
merge_usage_store_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_usage_store_user.pkl')
# 가입연도 추가
merge_usage_store_user['결제일자_연도'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.year
merge_usage_store_user['결제일자_월'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.month

merge_usage_store_user = merge_usage_store_user[merge_usage_store_user['결제일자_연도'] == 2022]

merge_usage_store_user = merge_usage_store_user[['가맹점ID', '가맹점명', '업종', '시도명', '시군구명', '회원번호', '성별', '연령대', '결제일자_연도', '결제일자_월', '결제금액']]

# 지역별 결제회원 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '회원번호']].drop_duplicates('회원번호')
merge_usage_store_user01 = merge_usage_store_user01.value_counts(['시도명', '시군구명']).reset_index()
merge_usage_store_user01.rename(columns = {'count' : '결제회원'}, inplace = True)


# 지역별 결제금액 생성
merge_usage_store_user02 = merge_usage_store_user.copy()
merge_usage_store_user02 = merge_usage_store_user02[['시도명', '시군구명', '결제금액']].\
                           groupby(['시도명', '시군구명']).agg({'sum'}).reset_index()
merge_usage_store_user02.columns = merge_usage_store_user02.columns.droplevel(1)
merge_usage_store_user02.columns = ['시도명', '시군구명', '결제금액']

# 결제회원 결제금액 결합
merge_view_df_buy02 = merge_usage_store_user01.merge(merge_usage_store_user02, on = ['시도명', '시군구명'])
merge_view_df_buy02['1인당_결제금액'] = merge_view_df_buy02['결제금액'] / merge_view_df_buy02['결제회원']
merge_view_df_buy02 = merge_view_df_buy02.astype({'1인당_결제금액' : 'int64', '결제금액' : 'int64'})

# 구매 및 결제금액 생성
merge_view_df_buy03 = merge_view_df_buy01.merge(merge_view_df_buy02, on = ['시도명', '시군구명'])

# 지역별 구매내역 및 결제내역 결과 시각화
merge_area_df = area_df.merge(merge_view_df_buy03, on = '시군구명')
merge_area_df = merge_area_df[['시도명', '시군구명', '구매회원', '구매금액', '1인당_구매금액', '결제회원', '결제금액', '1인당_결제금액', 'geometry']]
merge_area_df.to_file('../P3_Result/03.SHP/지역별_구매_결제내역_2022.gpkg')

In [ ]:
# 상품권 결제 정보 불러오기
merge_usage_store_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_usage_store_user.pkl')
# 가입연도 추가
merge_usage_store_user['결제일자_연도'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.year
merge_usage_store_user['결제일자_월'] = pd.to_datetime(merge_usage_store_user['결제일자']).dt.month

merge_usage_store_user = merge_usage_store_user[merge_usage_store_user['결제일자_연도'] == 2022]

merge_usage_store_user = merge_usage_store_user[['가맹점ID', '가맹점명', '업종', '시도명', '시군구명', '회원번호', '성별', '연령대', '결제일자_연도', '결제일자_월', '결제금액']]

# 지역별 업종당 가맹점 개수 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '업종', '가맹점ID']].drop_duplicates('가맹점ID')
merge_usage_store_user01 = merge_usage_store_user01.value_counts(['시도명', '시군구명', '업종']).reset_index()
merge_usage_store_user01.rename(columns = {'count' : '가맹점수'}, inplace = True)

# 업종기준 컬럼 카운트 갯수 추가
pivot_df01 = pd.pivot_table(merge_usage_store_user01, index = ['시도명', '시군구명'],\
                                                    columns = ['업종'],\
                                                     values = ['가맹점수']).reset_index()

# 컬럼명 수정
pivot_df01.columns = pivot_df01.columns.droplevel(1)
text = ['개인서비스업', '교육서비스업', '기타', '보건업', '소매업', '숙박업', '스포츠및여가관련서비스업', '여성청소년생필품', '음식점업', '제조업', '평생학습기관']
pivot_df01.columns = ['시도명', '시군구명'] + text
pivot_df01 = pivot_df01.fillna(0)
pivot_df01[text] = pivot_df01[text].astype(int)

# 지역별 구매내역 및 결제내역 결과 시각화
merge_area_df = area_df.merge(pivot_df01, on = '시군구명')
merge_area_df = merge_area_df[['시도명', '시군구명'] + text + ['geometry']]
merge_area_df.to_file('../P3_Result/03.SHP/지역별_업종별_가맹점수_2022.gpkg')

In [ ]:
# 지역별 업종별 결제금액 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '업종', '회원번호']].\
                           groupby(['시도명', '시군구명', '업종']).agg({'count'}).reset_index()
merge_usage_store_user01.columns = merge_usage_store_user01.columns.droplevel(1)
merge_usage_store_user01.columns = ['시도명', '시군구명', '업종', '업종별_결제회원']

# 업종기준 컬럼 카운트 갯수 추가
pivot_df01 = pd.pivot_table(merge_usage_store_user01, index = ['시도명', '시군구명'],\
                                                    columns = ['업종'],\
                                                     values = ['업종별_결제회원']).reset_index()

# 컬럼명 수정
pivot_df01.columns = pivot_df01.columns.droplevel(1)
text = ['개인서비스업', '교육서비스업', '기타', '보건업', '소매업', '숙박업', '스포츠및여가관련서비스업', '여성청소년생필품', '음식점업', '제조업', '평생학습기관']
pivot_df01.columns = ['시도명', '시군구명'] + text
pivot_df01 = pivot_df01.fillna(0)
pivot_df01[text] = pivot_df01[text].astype(int)

# 지역별 구매내역 및 결제내역 결과 시각화
merge_area_df = area_df.merge(pivot_df01, on = '시군구명')
merge_area_df = merge_area_df[['시도명', '시군구명'] + text + ['geometry']]
merge_area_df.to_file('../P3_Result/03.SHP/지역별_업종별_결제회원_2022.gpkg')

In [ ]:
# 지역별 업종별 결제금액 생성
merge_usage_store_user01 = merge_usage_store_user.copy()
merge_usage_store_user01 = merge_usage_store_user01[['시도명', '시군구명', '업종', '결제금액']].\
                           groupby(['시도명', '시군구명', '업종']).agg({'sum'}).reset_index()
merge_usage_store_user01.columns = merge_usage_store_user01.columns.droplevel(1)
merge_usage_store_user01.columns = ['시도명', '시군구명', '업종', '업종별_결제금액']

# 업종기준 컬럼 카운트 갯수 추가
pivot_df01 = pd.pivot_table(merge_usage_store_user01, index = ['시도명', '시군구명'],\
                                                    columns = ['업종'],\
                                                     values = ['업종별_결제금액']).reset_index()

# 컬럼명 수정
pivot_df01.columns = pivot_df01.columns.droplevel(1)
text = ['개인서비스업', '교육서비스업', '기타', '보건업', '소매업', '숙박업', '스포츠및여가관련서비스업', '여성청소년생필품', '음식점업', '제조업', '평생학습기관']
pivot_df01.columns = ['시도명', '시군구명'] + text
pivot_df01 = pivot_df01.fillna(0)
#pivot_df01[text] = pivot_df01[text].astype(int)

# 지역별 구매내역 및 결제내역 결과 시각화
merge_area_df = area_df.merge(pivot_df01, on = '시군구명')
merge_area_df = merge_area_df[['시도명', '시군구명'] + text + ['geometry']]
merge_area_df.to_file('../P3_Result/03.SHP/지역별_업종별_결제금액_2022.gpkg')

In [ ]:
# 업종별 결제비율 구하기
a = merge_area_df.iloc[:,2:-1]
a = a.div(a.sum(axis=1), axis=0)
merge_area_df.iloc[:,2:-1] = a
merge_area_df.to_file('../P3_Result/03.SHP/지역별_업종별_결제금액_비율_2022.gpkg')